# Clase 6 · ¿A quién llamamos? Clasificación con regresión logística

**Machine Learning** · Programa de Especialización Data Scientist · Módulo 2 · Sesión 6 de 15

Docente: **Josef Renato Rodriguez**

---

## El caso

Un banco portugués vende depósitos a plazo por teléfono. Entre mayo de 2008 y noviembre de 2010 hizo
41 188 llamadas. Con la crisis de 2008 recortó las campañas, y en 2011 quiere volver a escalarlas. No
puede llamar a todos: necesita un modelo que diga **a quién llamar primero**.

Los datos son reales: el estudio de Moro, Cortez y Rita (2014) sobre telemarketing bancario. Llegan como
llegan en un banco, **en cinco archivos de cinco áreas distintas**, y hay que pegarlos.

## Qué vas a hacer hoy, siguiendo CRISP-DM

| Bloque | Fase CRISP-DM | Contenido | Min |
|---|---|---|---|
| 1 | Entender el negocio | La decisión y **cuándo se conoce cada dato** | 4 |
| 2 | Entender y preparar los datos | La escalera: pegar 5 fuentes sin romper nada | 14 |
| 3 | Entender los datos | Explorar por período y cazar fugas de información | 9 |
| 4 | Modelar | Split aleatorio contra split temporal, y la regresión logística | 13 |
| 5 | Evaluar | Matriz de confusión, precision, recall, F1, ROC-AUC, KS y Gini | 14 |
| 6 | (adelanto) | El mundo cambia: drift y PSI | 6 |

Igual que siempre: completa las celdas **`TU CÓDIGO`** y ejecuta la **`VERIFICACIÓN`** de abajo.

---
## Celda 0 · Librerías y fuentes

Sube **`fuentes_banco.zip`** con el botón de archivos de Colab (el ícono de carpeta, a la izquierda). La celda
lo descomprime y carga los cinco archivos.

In [ ]:
import os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score)
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

if not os.path.exists("fuentes") and os.path.exists("fuentes_banco.zip"):
    zipfile.ZipFile("fuentes_banco.zip").extractall(".")
if not os.path.exists("fuentes"):
    raise FileNotFoundError("Sube fuentes_banco.zip al panel de archivos de Colab y vuelve a ejecutar esta celda.")

gestiones = pd.read_csv("fuentes/gestiones.csv", dtype={"cliente_id": str})
historial = pd.read_csv("fuentes/historial_campanas.csv")
clientes = pd.read_csv("fuentes/clientes.csv")
macro = pd.read_csv("fuentes/macro_mensual.csv")
respuestas = pd.read_csv("fuentes/respuestas.csv")

for nombre, df in [("gestiones", gestiones), ("historial", historial), ("clientes", clientes),
                   ("macro", macro), ("respuestas", respuestas)]:
    print(f"{nombre:>11}: {df.shape[0]:>6} filas · columnas: {list(df.columns)}")

---
# Bloque 1 · Entender el negocio  ·  4 min

**La decisión:** cada mes, el banco tiene una lista de clientes y capacidad para llamar a una parte. El
modelo ordena la lista por probabilidad de contratar, y se llama de arriba hacia abajo.

**La unidad:** una gestión, es decir, una llamada a un cliente en un mes. **El objetivo:** `contrato`
(1 si contrató el depósito, 0 si no).

**La pregunta que evita la mitad de los errores: ¿cuándo se conoce cada dato?** El modelo se usa
**antes** de llamar. Todo lo que se sabe recién durante o después de la llamada no puede entrar.

| Fuente | Qué trae | ¿Cuándo se conoce? |
|---|---|---|
| `gestiones` | canal, día, número de intentos, **duración de la llamada** | canal y día: antes. Intentos: durante. **Duración: al colgar** |
| `historial_campanas` | contactos en campañas anteriores y su resultado | antes de llamar |
| `clientes` | edad, ocupación, estado civil, créditos… | **una foto extraída en diciembre de 2010** |
| `macro_mensual` | euribor, empleo, precios, confianza | **al cerrar cada mes** |
| `respuestas` | si contrató | después de la campaña |

Guarda esta tabla: en el Bloque 3 vamos a encontrar las fugas que esconde.

---
# Bloque 2 · La escalera de pegado  ·  14 min

Pegar fuentes es una escalera: se parte de **la tabla que define la unidad** (las gestiones) y se sube un
peldaño a la vez. En cada peldaño se controlan dos cosas:

1. **Que no cambie el número de filas.** Si crece, la llave no era única y se duplicaron gestiones.
2. **Cuántos nulos aparecen, y por qué.** Un nulo después de un `left join` es un registro que no cruzó.

El argumento `validate` de `pd.merge` hace el control 1 por ti: si la relación no es la que dices, falla.

| `validate` | Significa |
|---|---|
| `"one_to_one"` | cada llave aparece una vez en cada tabla |
| `"many_to_one"` | la llave puede repetirse a la izquierda, pero es única a la derecha |

### Ejercicio 2.1 — Peldaño 1: gestiones + historial

Primero un detalle de la vida real: en `gestiones` el `cliente_id` viene como texto con ceros a la izquierda
(`"00512345"`), y en las otras fuentes como número. Así no cruzan nunca.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
base = gestiones.copy()
base["cliente_id"] = None       # convertir a entero: base["cliente_id"].astype(int)

paso1 = None                    # base.merge(historial, on=["cliente_id", "periodo"], how="left", validate="one_to_one")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("filas después del peldaño 1", None if paso1 is None else len(paso1), 41188, tol=0),
     check("columnas después del peldaño 1", None if paso1 is None else paso1.shape[1], 10, tol=0),
     check("gestiones sin historial", None if paso1 is None else paso1["contactos_previos"].isna().sum(), 0, tol=0)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### Ejercicio 2.2 — Peldaño 2: + clientes

El maestro tiene dos problemas, que no avisa nadie:

- **Clientes repetidos.** Algunos tienen una versión vieja (de antes de registrar su educación) y una nueva.
  Si no se limpia, el `merge` duplica sus gestiones. Nos quedamos con la versión más reciente según
  `fecha_actualizacion`.
- **Clientes que faltan.** Hay gestiones de clientes que no están en el maestro. El `left join` los conserva
  con nulos; `indicator=True` agrega la columna `_merge` para contarlos.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
n_repetidos = None              # cuántos cliente_id aparecen más de una vez: clientes["cliente_id"].duplicated().sum()
clientes_unicos = None          # clientes.sort_values("fecha_actualizacion").drop_duplicates("cliente_id", keep="last")

paso2 = None                    # paso1.merge(clientes_unicos, on="cliente_id", how="left", validate="many_to_one", indicator=True)
n_huerfanos = None              # cuántas filas de paso2 tienen _merge == "left_only"

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("clientes repetidos en el maestro", n_repetidos, 347, tol=0),
     check("clientes únicos", None if clientes_unicos is None else len(clientes_unicos), 40888, tol=0),
     check("filas después del peldaño 2", None if paso2 is None else len(paso2), 41188, tol=0),
     check("gestiones huérfanas", n_huerfanos, 300, tol=0)]
print()
print("2.2 OK" if all(r) else "Revisa 2.2")

Las 300 gestiones huérfanas (0.7 %) no se borran: quedan con nulos en las variables del
cliente, y el pipeline las rellena con «unknown», la misma etiqueta que el banco ya usa para lo que no sabe.
Lo importante es que **la decisión quede escrita**.

### Demostración 2.3 — Peldaño 3: + macro, con rezago

El promedio del euribor de marzo se conoce recién **al cerrar marzo**. Una llamada del 5 de marzo no puede
usarlo. La regla: cada gestión recibe **el último mes cerrado y publicado antes del mes de la llamada**.

Como la serie del banco tiene meses sin dato, «el mes anterior» no siempre existe. `pd.merge_asof` busca el
**último disponible hacia atrás**.

In [ ]:
# ── DEMOSTRACIÓN 2.3 ─────────────────────────────────────────────────────
if paso2 is not None:
    paso2["fecha_llamada"] = pd.to_datetime(paso2["periodo"])
    macro["fecha_publicacion"] = pd.to_datetime(macro["periodo"]) + pd.offsets.MonthBegin(1)   # se publica al iniciar el mes siguiente

    paso3 = pd.merge_asof(paso2.sort_values("fecha_llamada"),
                          macro.rename(columns={"periodo": "periodo_macro"}).sort_values("fecha_publicacion"),
                          left_on="fecha_llamada", right_on="fecha_publicacion", direction="backward")

    print(f"paso 3: {len(paso3)} filas · gestiones sin macro: {paso3['euribor_3m'].isna().sum()}\n")
    ejemplo = paso3.drop_duplicates("periodo")[["periodo", "periodo_macro", "euribor_3m"]]
    print(ejemplo[ejemplo["periodo"].isin(["2008-05", "2008-10", "2009-03", "2010-03"])].to_string(index=False))
else:
    paso3 = None

Las llamadas de octubre de 2008 usan el euribor de **agosto** (septiembre no está en la serie), y las de marzo
de 2009, el de **diciembre de 2008**. Usar el del mismo mes es una fuga temporal pequeña, pero real.

### Ejercicio 2.4 — Peldaño 4: + respuestas, y una variable nueva

`dias_ultimo_contacto` vale **999** cuando el cliente nunca fue contactado. Es un código, no un número: si
entra así, el modelo cree que 999 días es «muy lejos». Lo convertimos en una bandera.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
datos = None            # paso3.merge(respuestas, on=["cliente_id", "periodo"], how="left", validate="one_to_one")
# datos["contactado_antes"] = (datos["dias_ultimo_contacto"] != 999).astype(int)
# datos = datos.sort_values("gestion_id").reset_index(drop=True)

tasa_global = None      # la media de contrato

In [ ]:
# ── VERIFICACIÓN 2.4 ─────────────────────────────────────────────────────
r = [check("filas de la base final", None if datos is None else len(datos), 41188, tol=0),
     check("respuestas sin cruzar", None if datos is None else datos["contrato"].isna().sum(), 0, tol=0),
     check("tasa de contratación", tasa_global, 0.1127)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.4")

---
# Bloque 3 · Explorar por período y cazar fugas  ·  9 min

### Demostración 3.1 — Volumen y tasa, mes a mes

In [ ]:
# ── DEMOSTRACIÓN 3.1 ─────────────────────────────────────────────────────
if datos is not None:
    por_mes = datos.groupby("periodo").agg(llamadas=("contrato", "size"), tasa=("contrato", "mean"))
    fig, ax1 = plt.subplots(figsize=(11, 4))
    ax1.bar(por_mes.index, por_mes["llamadas"], color="#C5CEDF")
    ax1.set_ylabel("llamadas en el mes"); plt.xticks(rotation=70, fontsize=8)
    ax2 = ax1.twinx()
    ax2.plot(por_mes.index, por_mes["tasa"], "o-", color=MAG)
    ax2.set_ylabel("tasa de contratación", color=MAG); ax2.grid(False)
    ax1.set_title("Menos llamadas, más éxito: el banco cambió a quién llamaba")
    plt.tight_layout(); plt.show()
    print(datos.groupby(datos["periodo"].str[:4]).agg(llamadas=("contrato", "size"), tasa=("contrato", "mean"),
                                                       contactado_antes=("contactado_antes", "mean")).round(3))

- En 2008 el banco llamó 27 690 veces y contrató el 4.8 %. En 2010 llamó 2 058 veces y contrató el **52.1 %**.
- En 2010, el 35.2 % de los llamados ya había sido contactado antes; en 2008, casi nadie.

No es que la gente cambiara de golpe: **cambió la política de a quién llamar**. Después de la crisis, el banco
llamó poco y a clientes muy elegidos. Esto va a pesar en todo lo que sigue: el pasado no se parece al presente.

### Ejercicio 3.2 — El detector de fugas: AUC de una sola variable

Antes de modelar, mide cuánto separa cada variable **por sí sola**. Una variable que sola separa demasiado
bien es sospechosa. (El AUC lo vemos en el Bloque 5; por ahora: 0.5 es azar y 1.0 es separación perfecta.)

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
candidatas = ["edad", "n_llamadas", "contactos_previos", "contactado_antes", "duracion_seg",
              "n_productos_hoy", "euribor_3m", "n_empleados"]
auc_sola = {}
for c in (candidatas if datos is not None else []):
    v = datos[c].fillna(datos[c].median())
    a = None                    # roc_auc_score(datos["contrato"], v)
    if a is not None:
        auc_sola[c] = max(a, 1 - a)     # da igual si separa hacia arriba o hacia abajo

ranking = None                  # pd.Series(auc_sola).sort_values(ascending=False)

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
top2 = [] if ranking is None else list(ranking.index[:2])
r = [check_bool("las dos primeras son duracion_seg y n_productos_hoy", set(top2) == {"duracion_seg", "n_productos_hoy"}),
     check("AUC de duracion_seg sola", auc_sola.get("duracion_seg"), 0.8184)]
print()
print("3.2 OK" if all(r) else "Revisa 3.2")

### Las fugas del caso

Vuelve a la tabla del Bloque 1. Las dos primeras del ranking **no se conocen antes de llamar**, y hay una tercera más discreta:

| Fuga | Por qué | Qué hacemos |
|---|---|---|
| `duracion_seg` | la duración se sabe al colgar. Los que contratan hablan más: mediana de 449 s contra 164 s | **fuera** |
| `n_productos_hoy` | el maestro es una foto de diciembre de 2010: **ya incluye el depósito que vendió la campaña**. Promedio 3.0 contra 2.0 | **fuera** |
| `n_llamadas` | cuenta los intentos **de esta campaña**: al decidir a quién llamar, todavía no existe. Separa poco, pero la regla es la regla | **fuera** |
| macro del mismo mes | el promedio del mes se conoce al cerrarlo | rezago, en el peldaño 3 |
| split aleatorio | mezcla meses: entrena con el futuro para predecir el pasado | split temporal, en el Bloque 4 |

> Una fuga no es un error de código: el código corre perfecto y el modelo sale espectacular. Por eso es tan
> peligrosa. La única defensa es preguntar, variable por variable, **cuándo se conoce**.

Y las macro (`euribor_3m`, `n_empleados`) también separan bien, pero por otra razón: casi no cambian dentro de un
mes, así que **funcionan como un reloj**. Lo vemos en el bloque siguiente.

---
# Bloque 4 · Split aleatorio contra split temporal, y la regresión logística  ·  13 min

### La regresión logística en tres líneas

Predice una probabilidad. Calcula una combinación lineal, como la regresión de la Clase 2, y la pasa por la
**sigmoide**, que la aplasta entre 0 y 1:

$$z = b_0 + b_1 x_1 + \dots + b_p x_p \qquad\qquad \hat{p} = \frac{1}{1 + e^{-z}}$$

| Símbolo | Se lee | Qué es |
|---|---|---|
| $z$ | «zeta» | el puntaje lineal: los **log-odds** (logaritmo de las chances) |
| $\hat{p}$ | «pe sombrero» | la probabilidad de contratar |
| $e^{b_j}$ | «e a la be jota» | el **odds ratio**: cuánto se multiplican las chances si $x_j$ sube en una unidad |

### Los dos cortes

| Corte | Entrenamiento | Evaluación |
|---|---|---|
| Aleatorio 70/30 | 28 831 gestiones de todos los meses | 12 357 de todos los meses |
| Temporal | 36 939 · may 2008 – jun 2009 · tasa 7.3 % | validación jul – dic 2009 (2 191, tasa 39.8 %) · OOT 2010 (2 058, tasa 52.1 %) |

OOT es *out of time*: fuera de tiempo. Es lo más parecido a usar el modelo en el futuro.

### Ejercicio 4.1 — El corte temporal y el modelo

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
NUM = ["edad", "contactos_previos", "contactado_antes"]
CAT = ["ocupacion", "estado_civil", "educacion", "mora_previa", "hipoteca", "prestamo_consumo",
       "canal", "dia_semana", "resultado_previo"]
MACRO = ["euribor_3m", "var_empleo", "ipc", "confianza_consumidor", "n_empleados"]

def modelo(num, cat):
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="unknown"),
                              OneHotEncoder(handle_unknown="ignore")), cat)])
    return Pipeline([("prep", pre), ("logit", LogisticRegression(max_iter=2000))])

ent = None      # datos con periodo <= "2009-06"
val = None      # datos con periodo entre "2009-07" y "2009-12"
oot = None      # datos con periodo >= "2010-01"

logit = None    # modelo(NUM, CAT) entrenado con ent[NUM + CAT], ent["contrato"]
auc_val = None  # AUC en validación: roc_auc_score(val["contrato"], logit.predict_proba(val[NUM + CAT])[:, 1])
auc_oot = None  # AUC en OOT

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("gestiones de entrenamiento", None if ent is None else len(ent), 36939, tol=0),
     check("gestiones OOT", None if oot is None else len(oot), 2058, tol=0),
     check("AUC en validación", auc_val, 0.6194, tol=0.005),
     check("AUC en OOT", auc_oot, 0.7042, tol=0.005)]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Demostración 4.2 — La tabla que hay que mostrarle a cualquiera que reporte un AUC

El mismo modelo, con tres conjuntos de variables, medido con los dos cortes.

In [ ]:
# ── DEMOSTRACIÓN 4.2 ─────────────────────────────────────────────────────
if oot is not None:
    al_ent, al_test = train_test_split(datos, test_size=0.3, random_state=SEED, stratify=datos["contrato"])
    filas = []
    for nombre, num in [("sin macro", NUM), ("con macro", NUM + MACRO), ("con duración (fuga)", NUM + ["duracion_seg"])]:
        cols = num + CAT
        m_al = modelo(num, CAT).fit(al_ent[cols], al_ent["contrato"])
        m_t = modelo(num, CAT).fit(ent[cols], ent["contrato"])
        filas.append({"variables": nombre,
                      "AUC split aleatorio": roc_auc_score(al_test["contrato"], m_al.predict_proba(al_test[cols])[:, 1]),
                      "AUC validación temporal": roc_auc_score(val["contrato"], m_t.predict_proba(val[cols])[:, 1]),
                      "AUC OOT 2010": roc_auc_score(oot["contrato"], m_t.predict_proba(oot[cols])[:, 1])})
    print(pd.DataFrame(filas).set_index("variables").round(3).to_string())

Tres lecciones en una tabla:

1. **El split aleatorio promete de más.** Sin macro promete 0.727, y en el futuro el modelo da 0.619 y 0.704.
2. **Las macro engañan.** Con el split aleatorio suben el AUC a 0.786: el modelo aprende **en qué mes está** cada
   llamada, porque en el entrenamiento hay vecinos del mismo mes. Con el corte temporal **empeoran** el OOT
   (0.704 → 0.638): el euribor de 2010 no se parece a nada que haya visto.
3. **La fuga es espectacular.** Con la duración, el split aleatorio da 0.891. Ese modelo no se puede usar: la duración
   no existe antes de llamar.

Nos quedamos con el modelo **sin macro**, elegido con el corte temporal.

### Demostración 4.3 — Leer los coeficientes como odds ratios

In [ ]:
# ── DEMOSTRACIÓN 4.3 ─────────────────────────────────────────────────────
if logit is not None:
    nombres = [n.split("__")[1] for n in logit.named_steps["prep"].get_feature_names_out()]
    odds = pd.Series(np.exp(logit.named_steps["logit"].coef_[0]), index=nombres).sort_values()
    print("Multiplican las chances de contratar por…")
    print("\n  los que más SUBEN:");  print(odds.tail(5).round(2).to_string())
    print("\n  los que más BAJAN:");  print(odds.head(4).round(2).to_string())

- Ser **estudiante** multiplica las chances por 2.05, y ser **jubilado** por 1.62, frente al promedio de ocupaciones.
- Ser llamado a un **teléfono fijo** las multiplica por 0.48: casi la mitad que al celular.
- Que **la campaña anterior haya fracasado** con ese cliente, por 0.62.

Las numéricas están escaladas: su odds ratio es por **una desviación estándar** de cambio. Esta lectura es la
razón por la que la regresión logística sigue siendo el estándar en riesgos: cada efecto se puede explicar.

---
# Bloque 5 · Evaluar: las métricas de clasificación  ·  14 min

### La matriz de confusión

Con un umbral, cada probabilidad se convierte en «llamar» (1) o «no llamar» (0), y hay cuatro resultados:

| | predice 0 | predice 1 |
|---|---|---|
| **real 0** | VN · verdadero negativo | **FP** · falso positivo: llamada perdida |
| **real 1** | **FN** · falso negativo: venta perdida | VP · verdadero positivo |

| Métrica | Fórmula | Pregunta que responde |
|---|---|---|
| Exactitud | (VP + VN) / total | ¿qué proporción acertó? |
| **Precision** | VP / (VP + FP) | de los que llamé, ¿cuántos contrataron? |
| **Recall** | VP / (VP + FN) | de los que iban a contratar, ¿a cuántos llamé? |
| F1 | 2 · P · R / (P + R) | el balance entre las dos |

### Ejercicio 5.1 — La matriz con el umbral de 0.5, en el OOT

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
p_oot = None            # logit.predict_proba(oot[NUM + CAT])[:, 1]
pred_05 = None          # (p_oot >= 0.5).astype(int)
vn, fp, fn, vp = None, None, None, None     # confusion_matrix(oot["contrato"], pred_05).ravel()

precision = None        # precision_score(oot["contrato"], pred_05)
recall = None
f1 = None

In [ ]:
# ── VERIFICACIÓN 5.1 ─────────────────────────────────────────────────────
r = [check("verdaderos positivos", vp, 124, tol=15),
     check("precision", precision, 0.7702, tol=0.01),
     check("recall", recall, 0.1156, tol=0.01)]
print()
print("5.1 OK" if all(r) else "Revisa 5.1")

Precision alta (77.0 %) y recall bajísimo (11.6 %): el modelo solo marca «llamar» a 161 de 2058 clientes.
¿Por qué? Aprendió en una época en que contrataba el 7.3 %, y en promedio predice 0.22 cuando la tasa real es 0.52.
**Sus probabilidades quedaron descalibradas**, y el 0.5 deja de tener sentido.

> El umbral de 0.5 no es sagrado. En una campaña, el umbral lo fija **la capacidad**: si puedo llamar al 30 %
> de la lista, llamo al 30 % con mayor score. Con ese corte (umbral 0.34): precision 76.4 %, recall 44.0 %, F1 0.56.
> Elegir el umbral según costos lo vemos en la Clase 7.

### Métricas que no dependen del umbral: AUC, KS y Gini

| Métrica | Qué mide | Rango |
|---|---|---|
| **ROC-AUC** | probabilidad de que un contratante al azar tenga mayor score que un no contratante al azar | 0.5 azar · 1 perfecto |
| **KS** | máxima distancia entre las distribuciones acumuladas del score de los dos grupos = $\max(TPR - FPR)$ | 0 a 1 |
| **Gini** | $2 \cdot AUC - 1$ | 0 azar · 1 perfecto |

KS y Gini son los que se piden en riesgos de crédito y en los comités de modelos.

### Ejercicio 5.2 — AUC, KS y Gini en los tres tramos

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
def ks(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return None             # el máximo de (tpr - fpr)

resumen = {}
for nombre, df in ([("entrenamiento", ent), ("validación", val), ("OOT", oot)] if logit is not None else []):
    p = logit.predict_proba(df[NUM + CAT])[:, 1]
    auc = roc_auc_score(df["contrato"], p)
    resumen[nombre] = {"AUC": auc, "KS": ks(df["contrato"], p), "Gini": None}     # Gini = 2·AUC − 1

In [ ]:
# ── VERIFICACIÓN 5.2 ─────────────────────────────────────────────────────
o = resumen.get("OOT", {})
r = [check("KS en OOT", o.get("KS"), 0.3192, tol=0.01),
     check("Gini en OOT", o.get("Gini"), 0.4084, tol=0.01),
     check("KS en entrenamiento", resumen.get("entrenamiento", {}).get("KS"), 0.2211, tol=0.01)]
print()
print("5.2 OK" if all(r) else "Revisa 5.2")

### Demostración 5.3 — La curva ROC y la tabla por deciles

La tabla por deciles es como se presenta un modelo de campaña: se ordena la lista por score, se parte en 10
grupos iguales, y se mira la tasa real de cada grupo.

In [ ]:
# ── DEMOSTRACIÓN 5.3 ─────────────────────────────────────────────────────
if p_oot is not None:
    fpr, tpr, _ = roc_curve(oot["contrato"], p_oot)
    i_ks = np.argmax(tpr - fpr)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    axes[0].plot(fpr, tpr, color=BLUE, lw=2, label=f"modelo · AUC {roc_auc_score(oot['contrato'], p_oot):.3f}")
    axes[0].plot([0, 1], [0, 1], color=GRAY, ls="--", label="azar")
    axes[0].vlines(fpr[i_ks], fpr[i_ks], tpr[i_ks], color=MAG, lw=2, label=f"KS = {tpr[i_ks] - fpr[i_ks]:.3f}")
    axes[0].set_xlabel("FPR: no contratantes llamados"); axes[0].set_ylabel("TPR: contratantes llamados (recall)")
    axes[0].set_title("Curva ROC · OOT 2010"); axes[0].legend()

    decil = pd.qcut(pd.Series(p_oot).rank(method="first"), 10, labels=False)
    tabla_dec = pd.DataFrame({"y": oot["contrato"].values, "decil": 10 - decil}).groupby("decil")["y"].mean()
    axes[1].bar(tabla_dec.index, tabla_dec.values, color=[MAG if d == 1 else BLUE for d in tabla_dec.index])
    axes[1].axhline(oot["contrato"].mean(), color=GRAY, ls="--", label=f"tasa promedio {oot['contrato'].mean():.2f}")
    axes[1].set_xticks(range(1, 11)); axes[1].set_xlabel("decil de score (1 = los de mayor score)")
    axes[1].set_ylabel("tasa real de contratación"); axes[1].set_title("¿Ordena bien el modelo?"); axes[1].legend()
    plt.tight_layout(); plt.show()
    print(f"Lift del primer decil: {tabla_dec.loc[1] / oot['contrato'].mean():.2f} veces la tasa promedio")

El primer decil contrata el 76.7 % y el último el 20.9 %. Aunque las probabilidades estén descalibradas,
**el modelo ordena**: llamando primero a los de arriba, cada llamada rinde más. Para una campaña, ordenar bien
importa más que acertar la probabilidad exacta; para provisionar o fijar precios, no.

---
# Bloque 6 · Adelanto: el mundo cambia  ·  6 min

Un modelo aprende la foto del entrenamiento. Si la población cambia, el modelo sigue aplicando lo que aprendió
a gente distinta. A eso se le llama **drift** (deriva). El **PSI** (*Population Stability Index*) mide cuánto
cambió la distribución de una variable entre dos momentos:

$$PSI = \sum_{\text{tramos}} (\%_{actual} - \%_{esperado}) \cdot \ln\frac{\%_{actual}}{\%_{esperado}}$$

Los tramos son los deciles del entrenamiento (o las categorías). La convención habitual en la industria:

| PSI | Lectura |
|---|---|
| menos de 0.10 | estable |
| 0.10 a 0.25 | cambio moderado: vigilar |
| más de 0.25 | cambio fuerte: revisar el modelo |

### Demostración 6.1 — PSI entre el entrenamiento y 2010

In [ ]:
# ── DEMOSTRACIÓN 6.1 ─────────────────────────────────────────────────────
def psi(esperado, actual, tramos=10):
    esperado, actual = pd.Series(esperado).reset_index(drop=True), pd.Series(actual).reset_index(drop=True)
    if not pd.api.types.is_numeric_dtype(esperado) or esperado.nunique() <= tramos:      # categórica
        e = esperado.astype(str).value_counts(normalize=True)
        a = actual.astype(str).value_counts(normalize=True)
        cats = e.index.union(a.index)
        e, a = e.reindex(cats, fill_value=0), a.reindex(cats, fill_value=0)
    else:                                                                                 # numérica: deciles del esperado
        esperado, actual = esperado.dropna(), actual.dropna()
        cortes = np.unique(np.quantile(esperado, np.linspace(0, 1, tramos + 1)))
        cortes[0], cortes[-1] = -np.inf, np.inf
        e = pd.cut(esperado, cortes).value_counts(normalize=True, sort=False)
        a = pd.cut(actual, cortes).value_counts(normalize=True, sort=False)
    e, a = np.clip(e.values, 1e-4, None), np.clip(a.values, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))

if logit is not None and p_oot is not None:
    p_ent = logit.predict_proba(ent[NUM + CAT])[:, 1]
    filas = {c: psi(ent[c], oot[c]) for c in ["hipoteca", "canal", "edad", "ocupacion", "contactado_antes", "euribor_3m"]}
    filas["SCORE del modelo"] = psi(p_ent, p_oot)
    for c, v in filas.items():
        lectura = "estable" if v < 0.10 else ("vigilar" if v < 0.25 else "REVISAR")
        print(f"{c:>18}: PSI {v:>6.2f}  →  {lectura}")

La hipoteca no se movió (0.00): sirve de referencia. Casi todo lo demás sí: el canal (0.20), la edad (0.36), la ocupación (0.66), y sobre todo los clientes ya
contactados antes (1.39) y el euribor (más de 10: no hay superposición alguna). El **PSI del score** (1.25) resume todo:
el modelo está puntuando a una población que no se parece a la que lo entrenó.

Y aun así el AUC en 2010 fue razonable. **El PSI avisa que algo cambió; no dice si el modelo dejó de funcionar.**
Para eso hacen falta las respuestas. En la Clase 14 llegan los meses frescos de 2011: ahí vemos los tres tipos
de drift, el PSI completo y el piloto de la campaña contra un grupo de control.

---
# Cierre

### Los números del día

| | AUC | KS | Gini |
|---|---|---|---|
| Split aleatorio, sin macro | 0.727 | | |
| Split aleatorio, con la fuga de duración | 0.891 | | |
| **Temporal · validación 2009** | 0.619 | 0.211 | 0.239 |
| **Temporal · OOT 2010** | 0.704 | 0.319 | 0.408 |

### Checklist de salida

- [ ] Pego fuentes con `validate` y controlo filas y nulos en cada peldaño.
- [ ] Para cada variable me pregunto **cuándo se conoce**.
- [ ] Uso el AUC de una sola variable como detector de fugas.
- [ ] Cuando hay tiempo, corto por tiempo; el split aleatorio promete de más.
- [ ] Leo los coeficientes de la logística como odds ratios.
- [ ] Sé qué mide cada métrica de clasificación y que el umbral de 0.5 no es sagrado.
- [ ] Sé qué es el PSI y por qué no basta.

### Próxima clase · Miércoles 14/10

**Clase 7 · Árboles de clasificación, KNN y SVC, y el umbral de decisión.** Mismo caso: ¿puede un modelo no
lineal ordenar mejor? ¿Y dónde corto la lista si cada llamada cuesta y cada venta deja margen?